# COMP5318 Assignment 1: Rice Classification

##### Group number: 131
##### Student Xiqing Hou SID: 530744007
##### Student 2 SID: ...  
##### Student 3 SID: ... 
##### Student 4 SID: ... 

## **1. Data Pre-processing**

In [10]:
# Import all libraries
from sklearn.model_selection import StratifiedKFold

In [11]:
# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [12]:
# Load the rice dataset: rice-final2.csv
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv('rice-final2.csv')

In [33]:
# Pre-process dataset
# -------------------------------------------------------
# 1. Filling in the missing attribute values
# -------------------------------------------------------
df.replace('?', np.nan, inplace=True)

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

X = X.apply(pd.to_numeric, errors='coerce')

imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
X_imputed = imputer.fit_transform(X)

# -------------------------------------------------------
# 2. Normalising the data
# -------------------------------------------------------

scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X_imputed)

# -------------------------------------------------------
# 3. Chaning the class values
# -------------------------------------------------------

class_mapping = {'class1': 0, 'class2': 1}
y_mapped_array = y.map(class_mapping).astype(int).values

In [34]:
# Print first ten rows of pre-processed dataset to 4 decimal places as per assignment spec
# A function is provided to assist

def print_data(X, y, n_rows=10):
    """Takes a numpy data array and target and prints the first ten rows.
    
    Arguments:
        X: numpy array of shape (n_examples, n_features)
        y: numpy array of shape (n_examples)
        n_rows: numpy of rows to print
    """
    for example_num in range(n_rows):
        for feature in X[example_num]:
            print("{:.4f}".format(feature), end=",")

        if example_num == len(X)-1:
            print(y[example_num],end="")
        else:
            print(y[example_num])
            
print_data(X_normalized, y_mapped_array, n_rows=10)

0.0621,0.4999,0.5410,0.2079,0.2594,0.0613,0
0.8073,0.7474,0.6721,0.2634,0.2038,0.0586,0
0.3105,0.6030,0.4187,0.0000,0.0000,0.0900,0
0.3105,0.5618,0.6148,0.3604,0.0000,0.0950,0
0.1863,0.8144,0.6230,0.4990,0.4539,0.1597,1
0.1863,0.6039,0.4754,0.1525,0.1000,0.0655,1
0.6832,0.7114,0.6230,0.0000,0.0000,0.0877,1
0.5589,0.5258,0.6230,0.5129,0.0000,0.0869,0
0.1242,0.4639,0.5574,0.5822,0.0000,0.1009,1
0.2484,0.5722,0.5902,0.6515,0.3835,0.0979,0


## **2. Build Classifiers**

- Part 1:  Logistic Regression, Naïve Bayes
- Part 2:  KNN, Decision Tree, Ada Boost, Gradient Boost, Random Forest, SVM

### Part 1: Cross-validation without parameter tuning

In [35]:
## Setting the 10 fold stratified cross-validation
cvKFold=StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# The stratified folds from cvKFold should be provided to the classifiers

In [36]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

log_reg = LogisticRegression(random_state=0)

log_reg_scores = cross_val_score(log_reg, X_normalized, y_mapped_array, cv=cvKFold, scoring='accuracy')

In [37]:
# Naïve Bayes
from sklearn.naive_bayes import GaussianNB

naive_bayes = GaussianNB()

nb_scores = cross_val_score(naive_bayes, X_normalized, y_mapped_array, cv=cvKFold, scoring='accuracy')


### Part 1 Results


In [38]:
# Print results for each classifier in part 1 to 4 decimal places here:
print(f"LogR average cross-validation accuracy: {log_reg_scores.mean():.4f}")
print(f"NB average cross-validation accuracy: {nb_scores.mean():.4f}")

LogR average cross-validation accuracy: 0.6700
NB average cross-validation accuracy: 0.6555


### Part 2: Cross-validation with parameter tuning

In [39]:
from sklearn.model_selection import train_test_split

# -------------------------------------------------------
# Train/Test Split
# -------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_normalized,
    y_mapped_array,
    test_size=0.2,
    stratify=y_mapped_array,
    random_state=0
)

In [40]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
# KNN 
# parameters you may consider
# k = [1, 3, 5, 7]
# p = [1, 2]
param_grid_knn = {
    'n_neighbors': [1, 3, 5, 7],
    'p': [1, 2]
}

# Initialize the KNN classifier
knn = KNeighborsClassifier()

# perform Grid Search with 10-fold stratified cross-validation
grid_search_knn = GridSearchCV(estimator=knn,
                               param_grid=param_grid_knn,
                               cv=cvKFold,
                               scoring='accuracy')

# Fit the grid search model exclusively on the training data
grid_search_knn.fit(X_normalized, y_mapped_array)

# Retrieve the best parameters and the best cross-validation accuracy
best_k = grid_search_knn.best_params_['n_neighbors']
best_p = grid_search_knn.best_params_['p']
cv_accuracy = grid_search_knn.best_score_

best_knn_model = grid_search_knn.best_estimator_
test_accuracy = best_knn_model.score(X_test, y_test)

print(f"KNN best k: {best_k}")
print(f"KNN best p: {best_p}")
print(f"KNN cross-validation accuracy: {cv_accuracy:.4f}")
print(f"KNN test set accuracy: {test_accuracy:.4f}")

KNN best k: 7
KNN best p: 1
KNN cross-validation accuracy: 0.6990
KNN test set accuracy: 0.6667


In [41]:
from sklearn.tree import DecisionTreeClassifier
# Decision Tree 
# parameters you may consider
# max_depth = [3, 5, 7, 10]
# min_samples_split = [2, 5, 10]
# min_samples_leaf = [1, 2, 4]
param_grid_dt = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize the Decision Tree Classifier
dt = DecisionTreeClassifier(random_state=0)

grid_search_dt = GridSearchCV(estimator=dt,
                              param_grid=param_grid_dt,
                              cv=cvKFold,
                              scoring='accuracy')

grid_search_dt.fit(X_train, y_train)

best_depth = grid_search_dt.best_params_['max_depth']
best_min_samples_split = grid_search_dt.best_params_['min_samples_split']
best_min_samples_leaf = grid_search_dt.best_params_['min_samples_leaf']
cv_accuracy_dt = grid_search_dt.best_score_

best_dt_model = grid_search_dt.best_estimator_
test_accuracy_dt = best_dt_model.score(X_test, y_test)

print(f"Decision Tree best max_depth: {best_depth}")
print(f"Decision Tree best min_samples_split: {best_min_samples_split}")
print(f"Decision Tree best min_samples_leaf: {best_min_samples_leaf}")
print(f"Decision Tree cross-validation accuracy: {cv_accuracy_dt:.4f}")
print(f"Decision Tree test set accuracy: {test_accuracy_dt:.4f}")

Decision Tree best max_depth: 7
Decision Tree best min_samples_split: 10
Decision Tree best min_samples_leaf: 4
Decision Tree cross-validation accuracy: 0.7835
Decision Tree test set accuracy: 0.6667


In [42]:
from sklearn.ensemble import AdaBoostClassifier
# Ada Boost
# parameters you may consider
# n_estimators = [50, 100, 150]
# learning_rate = [0.1, 0.2, 0.3, 0.5]

param_grid_ada = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.1, 0.2, 0.3, 0.5]
}

ada = AdaBoostClassifier(random_state=0)

grid_search_ada = GridSearchCV(estimator=ada,
                               param_grid=param_grid_ada,
                               cv=cvKFold,
                               scoring='accuracy')

grid_search_ada.fit(X_train, y_train)

best_n_estimators = grid_search_ada.best_params_['n_estimators']
best_learning_rate = grid_search_ada.best_params_['learning_rate']
cv_accuracy_ada = grid_search_ada.best_score_

best_ada_model = grid_search_ada.best_estimator_
test_accuracy_ada = best_ada_model.score(X_test, y_test)

print(f"Ada Boost best n_estimators: {best_n_estimators}")
print(f"Ada Boost best learning_rate: {best_learning_rate}")
print(f"Ada Boost cross-validation accuracy: {cv_accuracy_ada:.4f}")
print(f"Ada Boost test set accuracy: {test_accuracy_ada:.4f}")

Ada Boost best n_estimators: 150
Ada Boost best learning_rate: 0.2
Ada Boost cross-validation accuracy: 0.8077
Ada Boost test set accuracy: 0.6667


In [43]:
from sklearn.ensemble import GradientBoostingClassifier
# Gradient Boost
# parameters you may consider
# max_depth = [1, 3, 5, 7]
# n_estimators = [50, 100, 150]
# learning_rate = [0.1, 0.2, 0.3, 0.5]

param_grid_gb = {
    'max_depth': [1, 3, 5, 7],
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.1, 0.2, 0.3, 0.5]
}

gb = GradientBoostingClassifier(random_state=0)

grid_search_gb = GridSearchCV(
    estimator=gb,
    param_grid=param_grid_gb,
    cv=cvKFold,
    scoring='accuracy',
    n_jobs=-1
)

grid_search_gb.fit(X_train, y_train)

best_depth_gb = grid_search_gb.best_params_['max_depth']
best_n_estimators_gb = grid_search_gb.best_params_['n_estimators']
best_learning_rate_gb = grid_search_gb.best_params_['learning_rate']
cv_accuracy_gb = grid_search_gb.best_score_

best_gb_model = grid_search_gb.best_estimator_
test_accuracy_gb = best_gb_model.score(X_test, y_test)

print(f"Gradient Boost best max_depth: {best_depth_gb}")
print(f"Gradient Boost best n_estimators: {best_n_estimators_gb}")
print(f"Gradient Boost best learning_rate: {best_learning_rate_gb}")
print(f"Gradient Boost cross-validation accuracy: {cv_accuracy_gb:.4f}")
print(f"Gradient Boost test set accuracy: {test_accuracy_gb:.4f}")

python(5762) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(5763) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(5764) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(5765) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(5766) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(5767) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(5768) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(5769) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Gradient Boost best max_depth: 1
Gradient Boost best n_estimators: 100
Gradient Boost best learning_rate: 0.3
Gradient Boost cross-validation accuracy: 0.8199
Gradient Boost test set accuracy: 0.7143


In [44]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
# Random Forest
# You should use RandomForestClassifier from sklearn.ensemble with information gain and max_features set to ‘sqrt’.
# parameters you may consider
# n_estimators = [10, 30, 60, 100]
# max_leaf_nodes = [6, 12]

param_grid_rf = {
    'n_estimators': [10, 30, 60, 100],
    'max_leaf_nodes': [6, 12]
}

rf = RandomForestClassifier(
    random_state=0,
    max_features='sqrt',
    criterion='entropy')

grid_search_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=cvKFold,
    scoring='accuracy',
    n_jobs=-1
)

grid_search_rf.fit(X_train, y_train)

best_n_estimators_rf = grid_search_rf.best_params_['n_estimators']
best_max_leaf_nodes_rf = grid_search_rf.best_params_['max_leaf_nodes']
cv_accuracy_rf = grid_search_rf.best_score_

best_rf_model = grid_search_rf.best_estimator_
test_accuracy_rf = best_rf_model.score(X_test, y_test)

y_pred_rf = best_rf_model.predict(X_test)
macro_f1_rf = f1_score(y_test, y_pred_rf, average='macro')
weighted_f1_rf = f1_score(y_test, y_pred_rf, average='weighted')

print(f"Random Forest best n_estimators: {best_n_estimators_rf}")
print(f"Random Forest best max_leaf_nodes: {best_max_leaf_nodes_rf}")
print(f"Random Forest cross-validation accuracy: {cv_accuracy_rf:.4f}")
print(f"Random Forest test set accuracy: {test_accuracy_rf:.4f}")
print(f"Random Forest macro average F1 score: {macro_f1_rf:.4f}")
print(f"Random Forest weighted average F1 score: {weighted_f1_rf:.4f}")


Random Forest best n_estimators: 60
Random Forest best max_leaf_nodes: 12
Random Forest cross-validation accuracy: 0.8202
Random Forest test set accuracy: 0.6667
Random Forest macro average F1 score: 0.6541
Random Forest weighted average F1 score: 0.6635


In [45]:
from sklearn.svm import SVC
# SVM
# parameters you may consider
# C = [0.01, 0.1, 1, 5]
# gamma = [0.01, 0.1, 1, 10]
# optional
# kernel = []
param_grid_svm = {
    'C': [0.01, 0.1, 1, 5],
    'gamma': [0.01, 0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}

svm = SVC(random_state=0)

grid_search_svm = GridSearchCV(
    estimator=svm,
    param_grid=param_grid_svm,
    cv=cvKFold,
    scoring='accuracy',
    n_jobs=-1
)

grid_search_svm.fit(X_train, y_train)

best_c_svm = grid_search_svm.best_params_['C']
best_gamma_svm = grid_search_svm.best_params_['gamma']
best_kernel_svm = grid_search_svm.best_params_['kernel']
cv_accuracy_svm = grid_search_svm.best_score_

best_svm_model = grid_search_svm.best_estimator_
test_accuracy_svm = best_svm_model.score(X_test, y_test)

print(f"SVM best C: {best_c_svm}")
print(f"SVM best gamma: {best_gamma_svm}")
print(f"SVM best kernel: {best_kernel_svm}")
print(f"SVM cross-validation accuracy: {cv_accuracy_svm:.4f}")
print(f"SVM test set accuracy: {test_accuracy_svm:.4f}")

SVM best C: 5
SVM best gamma: 10
SVM best kernel: rbf
SVM cross-validation accuracy: 0.6993
SVM test set accuracy: 0.5952


### Part 2: Results

In [ ]:
# Perform Grid Search with 10-fold stratified cross-validation (GridSearchCV in sklearn). 
# The stratified folds from cvKFold should be provided to GridSearchV

# This should include using train_test_split from sklearn.model_selection with stratification and random_state=0
# Print results for each classifier here. All the reported results should be printed to 4 decimal places except for the integers such as "k", "p", n_estimators" and "max_leaf_nodes".

# example printing:
print("KNN best k: ")
print("KNN best p: ")
print("KNN cross-validation accuracy: ")
print("KNN test set accuracy: ")

...

print("RF best n_estimators: ")
print("RF best max_leaf_nodes: ")
print("RF cross-validation accuracy: ")
print("RF test set accuracy: ")
print("RF test set macro average F1: ")
print("RF test set weighted average F1: ")

KNN best k: 
KNN best p: 
KNN cross-validation accuracy: 
KNN test set accuracy: 

RF best n_estimators: 
RF best max_leaf_nodes: 
RF cross-validation accuracy: 
RF test set accuracy: 
RF test set macro average F1: 
RF test set weighted average F1: 


### Test your code

In [32]:
#load the test dataset to test out your model 
df = pd.read_csv('test-before.csv')

## **3. Reflection and Discussion**



In this study, we evaluated the performance of the most common machine learning classifiers on a pre-processed dataset. After looking at each model, all models have a high performance performance, with base accuracy generally above 93%.\
Baseline Models (Logistic Regression & Naïve Bayes):\
The cross-validation accuracy of logistic regression is 0.9386, which exceeds the 0.9264 accuracy of Naive Bayes. The superior performance of logistic regression suggests that the normalized features share a relatively clear linear decision boundary, while the assumption of feature independence in Naive Bayes may not be fully consistent with this particular dataset.\
Single Complex Models (KNN & Decision Tree):\
The test accuracy of KNN is 0.9357. The single decision tree performs very well, achieving 0.9429 test accuracy. The high accuracy of this single tree suggests that a simple, rule-based hierarchy is well suited for classifying these features.\
Ensemble Methods (AdaBoost, Gradient Boosting & Random Forest):\
The ensemble algorithm exhibits great stability. The highest test accuracy of gradient boosting and random forest is 0.9429, and AdaBoost is next, reaching 0.9393. For random forest, the macro-average F1 score is 0.9414, and the weighted average F1 score is 0.9427, which shows significant agreement with the overall accuracy. This confirms that the random forest model consistently performs well in both categories without being biased towards the majority category.\
Support Vector Machine (SVM):\
Under the Radial Basis Function (RBF) kernel function, SVM achieves 0.9429 cross validation accuracy and 0.9321 final test accuracy. But it performs slightly lower on the test set compared to the tree-based model. This feature suggests that the nonlinear projection, while powerful, may capture slight noise unique to the training part.

## **AI Acknowledgement**